In [ ]:
import unittest
import re
from typing import List, Optional, Union, Dict, Any

# Import AST nodes and parser from your files
from ast_config import (
    ASTNode, KeYmaeraModel, ArchiveEntry, DefinitionsBlock, Definition, FuncDef, PredDef, HPDef,
    ProgramVariablesBlock, ProgramVariableDecl, ProblemBlock, TacticBlock,
    Term, Literal, VarRef, BinaryTermOp, UnaryTermOp, FunctionCall,
    Formula, BooleanLiteral, BinaryFormulaOp, UnaryFormulaOp, Comparison, Quantifier,
    ModalFormula, PredicateCall, C_Formula,
    HybridProgram, Assignment, NondetAssignment, Test, Sequence, Choice, Iteration,
    ODE, ODEEquation, HPBlock
)
from ast_parser import KeYmaeraParser # Also import parse_annotations if you want to test it separately

class TestKeYmaeraModalParsing(unittest.TestCase):

    def setUp(self):
        # MODIFIED: Removed space between 'waypoint' and 'EndDist' in the Problem block
        self.modal_string = """
Theorem "IJRR17/Theorem 14: Reach waypoint with deadline"

/*
 *	Robot must stop within distance delta at goal.
 *
 *  Robot
 * 	- must stop within distance delta of goal
 *  - can only drive straight and forward
 *  - ensures progress towards goal
 *
 *  Liveness property:
 *  - Robot can stop at goal
 *
 */

Definitions
    // @const: bound=[0.1:0.5]; def="time limit for control decisions"
    Real ep;      /* time limit for control decisions */
    // @const: bound=[0.5:5.0]; def="minimum braking capability of the robot"
    Real b;       /* minimum braking capability of the robot */
    // @const: bound=[1.0:10.0]; def="maximum acceleration"
    Real A;       /* maximum acceleration -b <= a <= A */
    // @const: bound=[1.0:10.0]; def="goal area size"
    Real GDelta;	 /* goal area size */
    // @const: bound=[5.0:30.0]; def="maximum robot velocity"
    Real Vmax;    /* robot cannot go faster than this */

    Real waypointStartDist(Real xg) = xg-GDelta;
    Real waypointEndDist(Real xg)   = xg+GDelta;

    Real minV = A*ep;
    Real maxTravelTime(Real g) = waypointStartDist(g)/minV;
    Real stopTime(Real v)      = v/b;
    Real speedUpTime(Real v)   = ep-v/A;

    Real stopDist(Real v) = v^2/(2*b);
    Real accComp(Real v)  = ( (A/b + 1)*(A/2*ep^2 + ep*v) );

    Bool bounds() <-> (                                                                 
        A > 0                                                                                         /* Working engine */
        & b > 0                                                                                         /* Working brakes */
        & ep > 0                                                                              /* Controller reaction time */
        & Vmax >= 2*A*ep
        & GDelta > Vmax*ep + Vmax^2/(2*b)  /* waypoint is large enough that robot can start driving and still stop inside */
    );

    Bool initialState(Real xr, Real vr, Real xg, Real T) <-> (                      
        vr = 0
        & xr < waypointStartDist(xg)
        & T > ep + maxTravelTime(xg-xr) + ep + stopTime(Vmax)
    );

    Bool assumptions(Real xr, Real vr, Real xg, Real T) <->              
        bounds() & initialState(xr, vr, xg, T);

    // @safety: penalty=-1000, on_failure=truncate:status=failure
    Bool loopinv(Real xr, Real vr, Real xg, Real T) <-> ( // Added T to parameters as it's used in the invariant
        0 <= vr & vr <= Vmax & xr + stopDist(vr) < waypointEndDist(xg)
                & (waypointStartDist(xg) < xr -> (vr = 0 | T >= stopTime(vr)))
                & (xr <= waypointStartDist(xg) ->
                    (vr >= minV & T > maxTravelTime(xg-xr) + ep + stopTime(Vmax))
                    | (vr <= minV & T > speedUpTime(vr) + maxTravelTime(xg-xr) + ep + stopTime(Vmax)))
    );

    // @progress: reward=1000, on_success=terminate:status=success, on_failure=terminate:status=failure
    Bool ensure(Real xr, Real vr, Real xg) <-> (
        waypointStartDist(xg) < xr & vr = 0
    );

    HP controlFunc ::= {
        // Choice 1: If in the goal area, decide to brake
        { ? (xr > waypointStartDist(xg));  ar := -b; } 
        ++
        // Choice 2: If in the goal area AND already stopped, decide to stay stopped
        { ? (xr > waypointStartDist(xg) & vr = 0);  ar := 0; }
        ++
        // Choice 3: If outside the goal area AND can accelerate, decide to accelerate
        { ? (xr <= waypointStartDist(xg) & (xr + stopDist(vr) + accComp(vr) < waypointEndDist(xg) & vr+A*ep <= Vmax));  ar := A; }
        ++
        // Choice 4: If outside the goal area AND cannot accelerate (must coast), decide to coast
        { ? (xr <= waypointStartDist(xg) & !(xr + stopDist(vr) + accComp(vr) < waypointEndDist(xg) & vr+A*ep <= Vmax));  ar := 0; }
    };

    HP stepFunc ::= {xr' = vr, vr' = ar, t' = 1, T'=-1 & t <= ep & vr >= 0};

    HP dwwpdl ::= {
        {	{ controlFunc; t := 0; }
            { stepFunc; }
        }*@invariant(loopinv(xr, vr, xg, T))
    };

End.

ProgramVariables
    // @var: bound=[0:100], def="robot position: x"
    Real xr;
    // @var: bound=[0:20], def="robot translational velocity"
    Real vr;
    // @var: bound=[-10:10], def="robot translational acceleration"
    Real ar;
    // @var: bound=[0:100], def="goal position: x"
    Real xg;
    // @var: bound=[0:1], def="time elapsed in current control step"
    Real t;
    // @var: bound=[0:1000], def="global time remaining until deadline"
    Real T;
End.

Problem
    assumptions(xr, vr, xg, T)
        -> ( [ dwwpdl; ](xr < waypointEndDist(xg) & (T <= 0 -> ensure(xr, vr, xg))) // Safety postcondition
        & < dwwpdl; > ensure(xr, vr, xg) )                                        // Liveness postcondition
End.
        """
        self.parser = KeYmaeraParser(self.modal_string)

    def test_full_modal_string_parsing(self):
        model = self.parser.parse()

        self.assertIsInstance(model, KeYmaeraModel)
        self.assertEqual(len(model.entries), 1)
        entry = model.entries[0]

        self.assertEqual(entry.kind, "IJRR17")
        self.assertEqual(entry.name, "Theorem 14: Reach waypoint with deadline")
        self.assertEqual(len(entry.tactics), 0) # No tactics defined in this example

        # --- Check Definitions ---
        self.assertIsInstance(entry.definitions, DefinitionsBlock)
        # 5 Real vars (ep, b, A, GDelta, Vmax)
        # 8 Real funcs (waypointStartDist, waypointEndDist, minV, maxTravelTime, stopTime, speedUpTime, stopDist, accComp)
        # 5 Bool preds (bounds, initialState, assumptions, loopinv, ensure)
        # 3 HP defs (controlFunc, stepFunc, dwwpdl)
        # Total: 5 + 8 + 5 + 3 = 21 definitions
        self.assertEqual(len(entry.definitions.defs), 21) 

        # Expected Real variable declarations (parsed as FuncDef with body as VarRef)
        expected_real_vars = [
            FuncDef('ep', 'Real', VarRef('ep'), annotations={'const': {'bound': [0.1, 0.5], 'def': 'time limit for control decisions'}}),
            FuncDef('b', 'Real', VarRef('b'), annotations={'const': {'bound': [0.5, 5.0], 'def': 'minimum braking capability of the robot'}}),
            FuncDef('A', 'Real', VarRef('A'), annotations={'const': {'bound': [1.0, 10.0], 'def': 'maximum acceleration'}}),
            FuncDef('GDelta', 'Real', VarRef('GDelta'), annotations={'const': {'bound': [1.0, 10.0], 'def': 'goal area size'}}),
            FuncDef('Vmax', 'Real', VarRef('Vmax'), annotations={'const': {'bound': [5.0, 30.0], 'def': 'maximum robot velocity'}})
        ]
        for i, expected in enumerate(expected_real_vars):
            self.assertEqual(entry.definitions.defs[i], expected, f"Mismatch in Real var def {i}: {entry.definitions.defs[i]} vs {expected}")

        # Expected Real function definitions
        expected_real_funcs = [
            FuncDef('waypointStartDist', 'Real', BinaryTermOp('-', VarRef('xg'), VarRef('GDelta')), args=[VarRef('xg')]),
            FuncDef('waypointEndDist', 'Real', BinaryTermOp('+', VarRef('xg'), VarRef('GDelta')), args=[VarRef('xg')]),
            FuncDef('minV', 'Real', BinaryTermOp('*', VarRef('A'), VarRef('ep'))),
            FuncDef('maxTravelTime', 'Real', BinaryTermOp('/', FunctionCall('waypointStartDist', [VarRef('g')]), VarRef('minV')), args=[VarRef('g')]),
            FuncDef('stopTime', 'Real', BinaryTermOp('/', VarRef('v'), VarRef('b')), args=[VarRef('v')]),
            FuncDef('speedUpTime', 'Real', BinaryTermOp('-', VarRef('ep'), BinaryTermOp('/', VarRef('v'), VarRef('A'))), args=[VarRef('v')]),
            FuncDef('stopDist', 'Real', BinaryTermOp('/', BinaryTermOp('^', VarRef('v'), Literal(2.0)), BinaryTermOp('*', Literal(2.0), VarRef('b'))), args=[VarRef('v')]),
            FuncDef('accComp', 'Real', BinaryTermOp('*', BinaryTermOp('+', BinaryTermOp('/', VarRef('A'), VarRef('b')), Literal(1.0)), BinaryTermOp('+', BinaryTermOp('*', BinaryTermOp('/', VarRef('A'), Literal(2.0)), BinaryTermOp('^', VarRef('ep'), Literal(2.0))), BinaryTermOp('*', VarRef('ep'), VarRef('v')))), args=[VarRef('v')])
        ]
        for i, expected in enumerate(expected_real_funcs):
            self.assertEqual(entry.definitions.defs[i+5], expected, f"Mismatch in Real func def {i+5}: {entry.definitions.defs[i+5]} vs {expected}")

        # Expected Bool predicate definitions
        # CORRECTED PARENTHESIS BALANCING AND LEFT-ASSOCIATIVITY FOR '&'
        expected_bounds_pred = PredDef('bounds', 'Bool',
            BinaryFormulaOp('&', # Outermost '&'
                BinaryFormulaOp('&', # Second outermost '&'
                    BinaryFormulaOp('&', # Third outermost '&'
                        BinaryFormulaOp('&', # Fourth outermost '&'
                            Comparison('>', VarRef('A'), Literal(0.0)),
                            Comparison('>', VarRef('b'), Literal(0.0))
                        ), # End of 4th '&'
                        Comparison('>', VarRef('ep'), Literal(0.0))
                    ), # End of 3rd '&'
                    Comparison('>=', VarRef('Vmax'), BinaryTermOp('*', BinaryTermOp('*', Literal(2.0), VarRef('A')), VarRef('ep')))
                ), # End of 2nd '&'
                Comparison('>', VarRef('GDelta'), BinaryTermOp('+', BinaryTermOp('*', VarRef('Vmax'), VarRef('ep')), BinaryTermOp('/', BinaryTermOp('^', VarRef('Vmax'), Literal(2.0)), BinaryTermOp('*', Literal(2.0), VarRef('b')))))
            ) # End of outermost '&'
        )
        self.assertEqual(entry.definitions.defs[13], expected_bounds_pred, "Mismatch in bounds() predicate definition.")

        expected_initialState_pred = PredDef('initialState', 'Bool',
            BinaryFormulaOp('&',
                BinaryFormulaOp('&',
                    Comparison('=', VarRef('vr'), Literal(0.0)),
                    Comparison('<', VarRef('xr'), FunctionCall('waypointStartDist', [VarRef('xg')]))
                ),
                Comparison('>', VarRef('T'), BinaryTermOp('+', BinaryTermOp('+', BinaryTermOp('+', VarRef('ep'), FunctionCall('maxTravelTime', [BinaryTermOp('-', VarRef('xg'), VarRef('xr'))])), VarRef('ep')), FunctionCall('stopTime', [VarRef('Vmax')])))
            ),
            args=[VarRef('xr'), VarRef('vr'), VarRef('xg'), VarRef('T')]
        )
        self.assertEqual(entry.definitions.defs[14], expected_initialState_pred, "Mismatch in initialState() predicate definition.")

        expected_assumptions_pred = PredDef('assumptions', 'Bool',
            BinaryFormulaOp('&',
                PredicateCall('bounds', []),
                PredicateCall('initialState', [VarRef('xr'), VarRef('vr'), VarRef('xg'), VarRef('T')])
            ),
            args=[VarRef('xr'), VarRef('vr'), VarRef('xg'), VarRef('T')]
        )
        self.assertEqual(entry.definitions.defs[15], expected_assumptions_pred, "Mismatch in assumptions() predicate definition.")

        # loopinv is very complex, we'll just check its type and name and some key components
        self.assertIsInstance(entry.definitions.defs[16], PredDef)
        self.assertEqual(entry.definitions.defs[16].name, 'loopinv')
        self.assertEqual(entry.definitions.defs[16].type, 'Bool')
        self.assertEqual(entry.definitions.defs[16].args, [VarRef('xr'), VarRef('vr'), VarRef('xg'), VarRef('T')])
        self.assertEqual(entry.definitions.defs[16].annotations, {'safety': {'penalty': -1000, 'on_failure': 'truncate:status=failure'}})
        # Further deep check for loopinv body would be extremely verbose,
        # but the parser's formula parsing is tested independently.

        expected_ensure_pred = PredDef('ensure', 'Bool',
            BinaryFormulaOp('&',
                Comparison('<', FunctionCall('waypointStartDist', [VarRef('xg')]), VarRef('xr')),
                Comparison('=', VarRef('vr'), Literal(0.0))
            ),
            args=[VarRef('xr'), VarRef('vr'), VarRef('xg')],
            annotations={'progress': {'reward': 1000, 'on_success': 'terminate:status=success', 'on_failure': 'terminate:status=failure'}}
        )
        self.assertEqual(entry.definitions.defs[17], expected_ensure_pred, "Mismatch in ensure() predicate definition.")


        # Expected HP definitions
        # controlFunc
        control_func_hp_str_inner = """
        { ? (xr > waypointStartDist(xg));  ar := -b; } 
        ++
        { ? (xr > waypointStartDist(xg) & vr = 0);  ar := 0; }
        ++
        { ? (xr <= waypointStartDist(xg) & (xr + stopDist(vr) + accComp(vr) < waypointEndDist(xg) & vr+A*ep <= Vmax));  ar := A; }
        ++
        { ? (xr <= waypointStartDist(xg) & !(xr + stopDist(vr) + accComp(vr) < waypointEndDist(xg) & vr+A*ep <= Vmax));  ar := 0; }
        """
        expected_control_func_hpdef_body = HPBlock(program=KeYmaeraParser(control_func_hp_str_inner).parse_hybrid_program(control_func_hp_str_inner.strip()))
        self.assertEqual(entry.definitions.defs[18], HPDef('controlFunc', expected_control_func_hpdef_body), "Mismatch in controlFunc HP definition.")

        # stepFunc
        expected_step_func_program_ode = ODE(
            equations=[
                ODEEquation(VarRef('xr'), VarRef('vr')),
                ODEEquation(VarRef('vr'), VarRef('ar')),
                ODEEquation(VarRef('t'), Literal(1.0)),
                ODEEquation(VarRef('T'), UnaryTermOp('-', Literal(1.0)))
            ],
            domain_constraint=BinaryFormulaOp('&',
                Comparison('<=', VarRef('t'), VarRef('ep')),
                Comparison('>=', VarRef('vr'), Literal(0.0))
            )
        )
        expected_step_func_body = expected_step_func_program_ode
        self.assertEqual(entry.definitions.defs[19], HPDef('stepFunc', expected_step_func_body), "Mismatch in stepFunc HP definition.")

        # dwwpdl
        # The definition is `HP dwwpdl ::= { ... };`
        # The content inside the outermost `{}` is an iteration: `{ { controlFunc; t := 0; } { stepFunc; } }*@invariant(...)`
        # So, the body of the HPDef should be an HPBlock, whose program is the Iteration.
        
        # First, define the program that is iterated: `{ { controlFunc; t := 0; } { stepFunc; } }`
        # This itself is an HPBlock containing a Sequence of two HPBlocks.
        program_to_iterate = HPBlock(
            program=Sequence([
                HPBlock(
                    program=Sequence([
                        VarRef('controlFunc'),
                        Assignment(VarRef('t'), Literal(0.0))
                    ])
                ),
                HPBlock(
                    program=VarRef('stepFunc')
                )
            ])
        )

        # Then, define the Iteration node itself, which uses program_to_iterate
        iteration_node = Iteration(
            program=program_to_iterate,
            invariant=PredicateCall('loopinv', [VarRef('xr'), VarRef('vr'), VarRef('xg'), VarRef('T')])
        )

        # Finally, the HPDef body is an HPBlock that wraps this iteration_node
        expected_dwwpdl_hpdef_body = HPBlock(program=iteration_node)

        self.assertEqual(entry.definitions.defs[20], HPDef('dwwpdl', expected_dwwpdl_hpdef_body), "Mismatch in dwwpdl HP definition.")


        # --- Check ProgramVariables ---
        self.assertIsInstance(entry.program_variables, ProgramVariablesBlock)
        self.assertEqual(len(entry.program_variables.vars), 6)

        expected_program_vars = [
            ProgramVariableDecl('Real', 'xr', annotations={'var': {'bound': [0, 100], 'def': 'robot position: x'}}),
            ProgramVariableDecl('Real', 'vr', annotations={'var': {'bound': [0, 20], 'def': 'robot translational velocity'}}),
            ProgramVariableDecl('Real', 'ar', annotations={'var': {'bound': [-10, 10], 'def': 'robot translational acceleration'}}),
            ProgramVariableDecl('Real', 'xg', annotations={'var': {'bound': [0, 100], 'def': 'goal position: x'}}),
            ProgramVariableDecl('Real', 't', annotations={'var': {'bound': [0, 1], 'def': 'time elapsed in current control step'}}),
            ProgramVariableDecl('Real', 'T', annotations={'var': {'bound': [0, 1000], 'def': 'global time remaining until deadline'}})
        ]
        for i, expected in enumerate(expected_program_vars):
            self.assertEqual(entry.program_variables.vars[i], expected, f"Mismatch in ProgramVariable {i}: {entry.program_variables.vars[i]} vs {expected}")

        # --- Check Problem ---
        self.assertIsInstance(entry.problem, ProblemBlock)
        problem_formula_str = "assumptions(xr, vr, xg, T) -> ([dwwpdl;](xr < waypointEndDist(xg) & (T <= 0 -> ensure(xr, vr, xg))) & <dwwpdl;>ensure(xr, vr, xg))"
        expected_problem_formula = self.parser.parse_formula(problem_formula_str)
        self.assertEqual(entry.problem.formula, expected_problem_formula)

if __name__ == '__main__':
    unittest.main(argv=['first-arg-is-ignored'], exit=False)

F
FAIL: test_full_modal_string_parsing (__main__.TestKeYmaeraModalParsing.test_full_modal_string_parsing)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "C:\Users\ADMIN\AppData\Local\Temp\ipykernel_114632\125679048.py", line 301, in test_full_modal_string_parsing
    self.assertEqual(entry.definitions.defs[20], HPDef('dwwpdl', expected_dwwpdl_hpdef_body), "Mismatch in dwwpdl HP definition.")
    ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: HPDef[16 chars]body=HPBlock(
    program=Iteration(
      pro[389 chars]s={}) != HPDef[16 chars]body=Iteration(
    program=HPBlock(
      pro[345 chars]s={}) : Mismatch in dwwpdl HP definition.

----------------------------------------------------------------------
Ran 1 test in 0.006s

FAILED (failures=1)
